# Fase 2 · Protocolo de MAREA-2 con `A2_hgt` — Mac mini

**Decisión (2026-10-02, con el profesor guía):** la versión del paper es `A2_hgt` (sin colonias + transferencia horizontal parcial adaptativa), la que más se acerca a L-SHADE/jSO en la Fase H.
Sustituye a `sin_colonias` (tag `marea2-sc-frozen`) y por eso hay que repetir el protocolo completo: 71 problemas × 51 corridas × {D10, D50} × {5k, 50k, 500k}.

Código congelado en el tag `marea2-hgt-frozen` (mismo `marea2.py` que verificó 355/355 en la Fase H).
Salida: `resultados_experimento/MAREA2_A2_hgt_D{10,50}_FES{5000,50000,500000}.json`. Se puede interrumpir y reanudar (la corrida retoma donde quedó).

**Pasos:** 0) entorno y hashes · 1) chequeo de entorno y presupuesto · 2) 5k (minutos) · 3) 50k (~1 h) · 4) 500k (~10 h) · 5) revisar los JSON.
Copiar al mini: `marea2.py`, `experimento_config.py`, `correr_marea2.py`, `correr_marea.py` (tag `marea2-hgt-frozen`).
**Solo en el Mac mini.**

In [ ]:
import sys, os, json, math, hashlib, subprocess, opfunu

WORKERS = 20
CONFIG  = "A2_hgt"
ESPERADO = {
    "marea2.py":            "10d7448e59082adf4f464068646d74d25d8b8f54f5317f9eb4dfed32dacfe64b",
    "experimento_config.py": "ef9c5af25d46afa681276efeb5d0e25c68290404dc6093e8d8783b3554a914d1",
    "correr_marea2.py":     "53396de1a263cde3eec183d0b6e43eb3dffb14c0c525f02fe47f5c63dd294652",
    "correr_marea.py":      "64421093bf66e9234f8157d8adfcc118aecb37617e9d5db9f735cbed270b4e6b",
}

def correr(*args):
    """Ejecuta un script con el mismo Python del kernel y muestra la salida en vivo."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        if "pkg_resources" not in linea:
            print(linea, end="")
    return p.wait()

print("python :", sys.executable)
print("carpeta:", os.getcwd())
print("opfunu :", opfunu.__version__)
assert opfunu.__version__ == "1.0.4", "opfunu tiene que ser 1.0.4 (1.0.1 esta roto)"
for f, h in ESPERADO.items():
    real = hashlib.sha256(open(f, "rb").read()).hexdigest()
    print(f"{f:24s} {real[:16]}  {'OK' if real == h else 'DISTINTO'}")
    assert real == h, f"{f} no es la version del tag marea2-hgt-frozen: copiala de nuevo"
print("\nEntorno y archivos OK")

## 1 · Chequeo de entorno y presupuesto (segundos)

In [ ]:
rc = correr("correr_marea2.py", "--check", "--config", CONFIG)
assert rc == 0, "el chequeo fallo: no seguir, mandar la salida a Claude"
CHEQUEADO = True

## 2 · 5k (minutos)

In [ ]:
assert globals().get("CHEQUEADO"), "primero la celda de chequeo"
rc = correr("correr_marea2.py", "--config", CONFIG, "--maxfes", "5000", "--workers", str(WORKERS))
assert rc == 0

## 3 · 50k (~1 h)

In [ ]:
assert globals().get("CHEQUEADO"), "primero la celda de chequeo"
rc = correr("correr_marea2.py", "--config", CONFIG, "--maxfes", "50000", "--workers", str(WORKERS))
assert rc == 0

## 4 · 500k (~10 h; se puede dejar corriendo de noche)

In [ ]:
assert globals().get("CHEQUEADO"), "primero la celda de chequeo"
rc = correr("correr_marea2.py", "--config", CONFIG, "--maxfes", "500000", "--workers", str(WORKERS))
assert rc == 0

## 5 · Revisar y traer los JSON al MacBook
Los 6 `MAREA2_A2_hgt_*.json` van a `resultados_experimento/` del MacBook (**no a la raíz**).

In [ ]:
for fes in (5000, 50000, 500000):
    for d in (10, 50):
        p = f"resultados_experimento/MAREA2_{CONFIG}_D{d}_FES{fes}.json"
        if not os.path.exists(p):
            print(f"FALTA   {p}"); continue
        v = json.load(open(p))
        malos = sum(1 for x in v.values() if not math.isfinite(x) or x < 0)
        print(f"{'OK     ' if len(v) == 71*51 and not malos else 'REVISAR'} {os.path.basename(p)}: "
              f"{len(v)}/{71*51} corridas, {malos} no finitos o negativos")